# ImageNet-1K — analysis & experiments

Thin notebook: all logic lives in `vision_pipeline/`, shared with the
[CIFAR-10 notebook](cifar10_analysis.ipynb). The dataset is chosen by `cfg.data.dataset`.

| module | contents |
|---|---|
| `config.py` | config dataclasses, `Config.preset("cifar10"\|"imagenet"\|"imagenet_vit")`, `set_seed`, `get_device` |
| `data/cifar10.py`, `data/imagenet.py` | one file per dataset: transforms + dataset construction |
| `data/__init__.py` | dataset registry, `Loaders`, `build_loaders` |
| `models/resnet.py` | blocks + `ResNet` with a switchable stem (`cifar` / `imagenet`) |
| `models/vit.py` | `PatchEmbedding`, `MultiHeadSelfAttention`, `TransformerEncoderLayer`, `ViT` |
| `models/cnn.py` | `CNN` — the plain conv baseline |
| `models/__init__.py` | `build_model` registry |
| `train.py` | optimizer/scheduler/criterion, `check_accuracy`, `confusion_matrix`, `train`, `setup` |
| `inference.py` | `Predictor`, `evaluate_checkpoint`, TorchScript/ONNX export |
| `cli.py` | `python -m vision_pipeline.cli train\|eval\|predict --dataset imagenet` |

Long runs belong in the CLI (a notebook kernel leaks DataLoader workers on
interrupt — the source of the `worker exited unexpectedly` crashes):

```bash
python -m vision_pipeline.cli train --dataset imagenet --arch resnet50 --epochs 100
python -m vision_pipeline.cli train --preset imagenet_vit --epochs 100        # ViT-S/16
python -m vision_pipeline.cli train --dataset imagenet --resume checkpoints/<run>_best.pt
```

`--preset` picks the config recipe, `--dataset` picks the data; ViT needs its own
recipe (lower lr, wd 0.1, 10-epoch warmup, grad clip 1.0) on the same ImageNet data.

Data download notes: <https://velog.io/@jasonlee1995/Linux-Server-Download-ImageNet-1K>

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace

import matplotlib.pyplot as plt
import torch

from vision_pipeline import (
    Config, DataConfig, ModelConfig, TrainConfig,
    Predictor, build_loaders, build_model, check_accuracy, confusion_matrix,
    count_parameters, evaluate_checkpoint, get_device, normalization, set_seed,
    setup, top1, train,
)

device = get_device()
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

## 1. Configure

Two presets for this dataset:

* `Config.preset("imagenet")` — resnet34 + the ImageNet stem + AdamW 1e-3.
* `Config.preset("imagenet_vit")` — ViT-S/16 + AdamW 3e-4, weight_decay 0.1,
  10-epoch warmup, grad clip 1.0, RandomErasing 0.25.

AdamW is the optimizer for this dataset either way (SGD only won on CIFAR-10).
Flip `USE_VIT` below; override anything after that.

In [ ]:
# Flip this to switch the whole notebook between the ResNet and the ViT run.
USE_VIT = True

if USE_VIT:
    cfg = Config.preset("imagenet_vit")
    # vit_tiny / vit_small / vit_base, or arch="vit_custom" + explicit knobs:
    # cfg = replace(cfg, model=replace(cfg.model, arch="vit_custom",
    #                                  embed_dim=384, depth=8, num_heads=6, patch_size=16))
    cfg = replace(cfg, model=replace(cfg.model, arch="vit_small"))
else:
    cfg = Config.preset("imagenet")
    # resnet50 (bottleneck blocks) instead of the preset's resnet34.
    cfg = replace(cfg, model=replace(cfg.model, arch="resnet50"))

# Tweak further from here, e.g.:
# cfg = replace(cfg, train=replace(cfg.train, lr=3e-4))
# cfg = replace(cfg, data=replace(cfg.data, batch_size=256))

# ViT bakes resolution into pos_embed (one row per patch), so these must agree.
assert cfg.model.image_size == cfg.data.image_size
cfg

## 2. Data

In [ ]:
set_seed(cfg.data.seed)
loaders = build_loaders(cfg.data)
print(loaders.summary())

In [ ]:
# Sanity-check one batch: shapes, dtypes, label range.
images, targets = next(iter(loaders.train))
print("Image batch:", images.shape, images.dtype)
print("Target batch:", targets.shape, targets.dtype)
print("Target range:", targets.min().item(), targets.max().item())

In [ ]:
# Visualise the augmented batch (un-normalize first) — the fastest way to catch
# a broken transform pipeline. `normalization()` returns the dataset's mean/std.
mean_t, std_t = (torch.tensor(v).view(3, 1, 1) for v in normalization(cfg.data))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, target in zip(axes.ravel(), images, targets):
    ax.imshow((img * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(f"class {target.item()}", fontsize=8)
    ax.axis("off")
fig.tight_layout()

## 3. Model

In [ ]:
set_seed(cfg.train.seed)  # seed right before weight init so the run is reproducible
torch.backends.cudnn.benchmark = True  # fixed input size -> let cudnn pick fastest kernels

net = build_model(cfg.model, device=device)
print(f"{count_parameters(net) / 1e6:.2f}M parameters")

In [ ]:
# Forward pass on the real batch — catches stem/head shape mistakes early.
batch = images.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
net.eval()
with torch.inference_mode():
    logits = net(batch)

print("Output:", logits.shape)
assert logits.shape == (batch.size(0), cfg.model.num_classes)

In [ ]:
# Compare architectures cheaply — parameter counts only, no data needed.
# ResNets are resolution-agnostic (adaptive pool); the ViTs are built at
# cfg.model.image_size with cfg.model.patch_size, so token count is printed too.
for arch in ['resnet18', 'resnet34', 'resnet50', 'resnet101']:
    m = build_model(replace(cfg.model, arch=arch, stem="imagenet"))
    print(f"{arch:>10}: {count_parameters(m) / 1e6:6.2f}M")

for arch in ['vit_tiny', 'vit_small', 'vit_base']:
    m = build_model(replace(cfg.model, arch=arch, patch_size=16))
    print(f"{arch:>10}: {count_parameters(m) / 1e6:6.2f}M  "
          f"({m.num_patches} patches + 1 CLS, embed {m.embed_dim})")
del m

## 4. Train

`train()` returns a `History` with per-epoch curves (including `gap`,
train − held-out, the overfitting readout).

The cell below is a **full-length run** (100 epochs). For a quick end-to-end
check of the loop instead, shrink it: `epochs=1, warmup_epochs=0`.

In [ ]:
probe_cfg = replace(cfg, train=replace(cfg.train, epochs=100, warmup_epochs=5, run_name="probe"))

history = train(net, loaders, probe_cfg, device=device)
history.to_frame()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history.epoch, history.loss);                  axes[0].set_title("train loss")
axes[1].plot(history.epoch, history.train_acc, label="train")
axes[1].plot(history.epoch, history.test_acc, label="held-out"); axes[1].legend(); axes[1].set_title("accuracy")
axes[2].plot(history.epoch, history.gap);                   axes[2].set_title("train - held-out gap")
for ax in axes: ax.set_xlabel("epoch")
fig.tight_layout()

### Parameter sweep

Each variant gets a fresh model + its own run directory, so TensorBoard curves
never overlap. Keep `epochs` small — this is for ranking, not final numbers.
One ImageNet epoch is expensive, so treat even this as a rough signal.

In [ ]:
# One AdamW run per learning rate — everything else held fixed, so the
# difference in the curves is attributable to lr alone. ViTs live at a lower lr
# and a much higher weight decay than the ResNets, so the grid differs.
if USE_VIT:
    sweep = {
        "adamw_1e-3": replace(cfg.train, lr=1e-3, weight_decay=0.1),
        "adamw_3e-4": replace(cfg.train, lr=3e-4, weight_decay=0.1),
        "adamw_1e-4": replace(cfg.train, lr=1e-4, weight_decay=0.1),
    }
else:
    sweep = {
        "adamw_1e-3": replace(cfg.train, lr=1e-3, weight_decay=1e-2),
        "adamw_3e-4": replace(cfg.train, lr=3e-4, weight_decay=1e-2),
        "adamw_1e-4": replace(cfg.train, lr=1e-4, weight_decay=1e-2),
    }

results = {}
for name, tcfg in sweep.items():
    variant = replace(cfg, train=replace(tcfg, epochs=1, warmup_epochs=0, run_name=f"sweep_{name}"))
    set_seed(variant.train.seed)
    model = build_model(variant.model, device=device)
    results[name] = train(model, loaders, variant, device=device, verbose=False)
    print(f"{name:>12}: best held-out {results[name].best_acc:.4f}")
    del model
    torch.cuda.empty_cache()

In [ ]:
for name, h in results.items():
    plt.plot(h.epoch, h.test_acc, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("held-out top-1"); plt.legend(); plt.title("sweep")

In [ ]:
# Augmentation sweep — same knob shape, on DataConfig instead. Note this one
# rebuilds the loaders, since the transform is baked into the dataset.
# for scale in [(0.08, 1.0), (0.35, 1.0)]:
#     variant = replace(
#         cfg,
#         data=replace(cfg.data, crop_scale=scale),
#         train=replace(cfg.train, epochs=1, warmup_epochs=0, run_name=f"crop_{scale[0]}"),
#     )
#     set_seed(variant.data.seed)
#     ldrs = build_loaders(variant.data)
#     set_seed(variant.train.seed)
#     m = build_model(variant.model, device=device)
#     h = train(m, ldrs, variant, device=device, verbose=False)
#     print(f"crop_scale={scale}: best {h.best_acc:.4f}, final gap {h.gap[-1]:.4f}")

### Resume from a checkpoint

`setup()` builds model/loaders/optimizer/scheduler and restores state in one call.

In [ ]:
# CKPT = "checkpoints/20260722-150350_resnet34_best.pt"
# model, loaders, optimizer, scheduler, criterion, device, resume_state = setup(
#     cfg, resume=CKPT, fresh_schedule=False,
# )
# history = train(
#     model, loaders, cfg,
#     optimizer=optimizer, scheduler=scheduler, criterion=criterion,
#     device=device, **resume_state,
# )

## 5. Evaluate & inference

In [ ]:
# Top-1/top-5 on the held-out val split, and on the clean fixed balanced train
# subset (10 images per class) for the train-side reading.
print("val:  ", check_accuracy(loaders.val, net, device=device, topk=(1, 5)))
print("train:", check_accuracy(loaders.eval_train, net, device=device, topk=(1, 5)))

In [ ]:
# Where does it actually fail? A 1000x1000 grid is unreadable, so pull the
# worst-confused class pairs out of it instead of plotting the whole thing.
cm = confusion_matrix(loaders.eval_train, net, num_classes=cfg.model.num_classes, device=device)

off_diagonal = cm.clone()
off_diagonal.fill_diagonal_(0)
counts, flat = off_diagonal.flatten().topk(15)
for count, index in zip(counts.tolist(), flat.tolist()):
    true_label, pred_label = divmod(index, cfg.model.num_classes)
    print(f"{count:4d}  class {true_label:3d} -> predicted {pred_label:3d}")

In [ ]:
# Per-class accuracy — the tail is where ImageNet models actually lose points.
per_class = cm.diagonal().float() / cm.sum(1).clamp(min=1)
order = per_class.argsort()
print("worst 10:", [(int(i), round(float(per_class[i]), 3)) for i in order[:10]])
print("best 10: ", [(int(i), round(float(per_class[i]), 3)) for i in order[-10:]])

plt.hist(per_class.numpy(), bins=25)
plt.xlabel("per-class accuracy"); plt.ylabel("number of classes")

In [ ]:
# The checkpoint embeds its Config, so the Predictor rebuilds the right
# architecture AND the right eval transform on its own.
# CKPT = "checkpoints/20260722-150350_resnet34_best.pt"
# print(evaluate_checkpoint(CKPT, cfg, topk=(1, 5)))
# predictor = Predictor.from_checkpoint(CKPT)
# predictor.predict_paths(["dataset/sample.jpg"], topk=5)

In [ ]:
# Export for serving without the Python model code.
# predictor.export_torchscript("checkpoints/resnet50_imagenet.ts")
# predictor.export_onnx("checkpoints/resnet50_imagenet.onnx")

## Benchmark

### Goal

ImageNet-1K (single centre crop, 224x224) — published reference numbers, as a
yardstick for what these architectures can reach:

| arch | top-1 | top-5 | params | recipe |
|---|---|---|---|---|
| ResNet-18 | ~69.8% | ~89.1% | 11.7M | 90-epoch SGD |
| ResNet-34 | ~73.3% | ~91.4% | 21.8M | 90-epoch SGD |
| ResNet-50 | ~76.1% | ~92.9% | 25.6M | 90-epoch SGD |
| ViT-Ti/16 | ~72.2% | — | 5.7M | 300-epoch AdamW + heavy aug (DeiT) |
| ViT-S/16 | ~79.8% | — | 22.1M | 300-epoch AdamW + heavy aug (DeiT) |
| ViT-B/16 | ~77.9% | — | 86.6M | 300-epoch AdamW + heavy aug (DeiT) |

* Good top-1 for a from-scratch run in this setup: 65%+
* Gap (train − val) of 5%–10% is normal; ImageNet is large enough that
  memorization is much harder than on CIFAR-10.
* One epoch is ~1.28M images — expect minutes per epoch, not seconds. Budget
  the run before starting it.

**Reading the ViT rows honestly.** Those numbers come from the DeiT recipe:
300 epochs, Mixup/CutMix, RandAugment, stochastic depth, repeated augmentation
and EMA. This pipeline has RandomResizedCrop + HFlip + RandomErasing only, so a
100-epoch `imagenet_vit` run should be expected to land *well* below them —
plain ViT-B/16 in the original paper only reached ~77.9% *with* JFT-300M
pretraining, and trained from scratch on ImageNet-1K alone it lands in the high
60s / low 70s. The ResNet rows, by contrast, are reachable with roughly the
augmentation this pipeline already has. Same lesson as CIFAR-10 v3.0, one scale
up: no convolutional prior means the augmentation column carries the model.

Next augmentation levers, none implemented yet (in rough payoff order):
Mixup/CutMix → RandAugment → stochastic depth (drop_path) → EMA of weights.

### Fine-tuning results

No completed ImageNet run is recorded yet — the only logged attempt
(`resnet34`, AdamW 1e-3, 100 epochs) died at the start with
`DataLoader worker (pid(s) ...) exited unexpectedly`, which is the notebook
kernel losing its workers. Run it from the CLI instead:

```bash
python -m vision_pipeline.cli train --dataset imagenet --arch resnet50 --epochs 100
python -m vision_pipeline.cli train --preset imagenet_vit --epochs 100
python -m vision_pipeline.cli train --preset imagenet_vit --arch vit_tiny --batch-size 256
```

The only CIFAR-10 ViT datapoint so far (v3.0, `vit_simple`, 3.20M params):
best test 84.35% with train at 100.0% — a 15.6% gap, i.e. pure overfitting on
50k images. ImageNet-1K is 25x larger, so the same architecture family should
show a much smaller gap here; if a ViT run on ImageNet still shows a >10% gap,
the augmentation is the thing to fix, not the depth.

#### Template for recording a run

    #### v1.0 — <arch> (<date>)

        * Optimiser: AdamW lr=..., weight_decay=..., label_smoothing=0.1
        * Schedule: ...-epoch linear warmup -> cosine to 0, grad_clip=...
        * ViT only: patch_size=..., embed_dim=..., depth=..., num_heads=...,
          mlp_dim=..., drop_rate=...; cls_token/pos_embed excluded from weight decay
        * # epoch = ..., batch_size = ...
        * Augmentation: RandomResizedCrop(scale=...) + HFlip + RandomErasing(p=...)
        * Systems: AMP + channels_last + cudnn.benchmark; seed 42

    Result:

        Train Acc (balanced 10k subset): ...
        Val top-1: ...   Val top-5: ...   gap: ...
        ... s/epoch, ... total